## Load the summary information and preprocessed data

In [1]:
import os
import json
import pandas as pd

from pathlib import Path

from single_sample_gsea import ss_gsea


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 89 samples and 17014 columns.
First few rows of the data:


,overall survival (os) (month),outcome,samples,age,Sex,t stage,n stage,ajcc/uicc-tnm stage,OR4F5,OR4F29,...,COL6A2,FTCD,LSS,MCM3AP,YBEY,C21orf58,PCNT,DIP2A,S100B,PRMT2
0,34,1,CRC_1,63,Male,T4b,N0,IV,2.238720,3.576052,...,6.904159,0.513126,5.505524,4.826873,2.024319,1.362392,4.871521,4.314822,1.041236,3.873577
1,34,0,CRC_2,72,Female,T3,N2a,IV,2.690912,4.288482,...,6.167773,1.052961,4.406301,4.852319,2.342852,1.545884,5.215702,4.589853,1.031734,3.731309
2,33,0,CRC_3,58,Male,T4a,N2a,IV,2.906905,3.063055,...,7.235745,0.971923,4.432446,4.652795,1.590173,0.714812,5.113489,4.286788,1.482064,3.758579
3,32,0,CRC_4,73,Male,T4a,N2b,IV,3.284267,3.798752,...,6.739188,1.031439,4.180149,4.711043,1.433220,1.109589,4.941454,4.435966,1.507448,3.652840
4,28,1,CRC_5,66,Female,T3,N2a,IV,4.136243,5.140208,...,5.951511,1.995989,4.102097,4.297774,1.687317,1.307901,4.931410,4.270836,1.811508,3.717703


## Load gene sets

In [2]:
MSIGDB_JSON = Path("../../resources/h.all.v2026.1.Hs.json")

with MSIGDB_JSON.open(encoding="utf-8") as f:
    gene_sets = {name: data["geneSymbols"] for name, data in json.load(f).items()}

print(f"Loaded {len(gene_sets)} gene sets.")


Loaded 50 gene sets.


## Compute single sample GSEA on the preprocessed data

Select only gene columns for the computation.

In [3]:
# IMPORTANT: This is dataset-specific and is the only variable that needs to be changed for each of the dataset.
sample_id_column = 'samples'

gene_expression_df = df.iloc[:, summary_data['data_summary']['clinical_features']:].copy()
gene_expression_df.index = df[sample_id_column].astype(str)
gene_expression_df = gene_expression_df.T
gene_expression_df.index.name = "gene"

result = ss_gsea(gene_expression_df, gene_sets)


## Merge the results with the clinical data

The output table should have the same number of rows as before and columns should sum up to clinical data + 50 gene sets.

In [4]:
clinical_df = df.iloc[:, :summary_data["data_summary"]["clinical_features"]].copy()
clinical_df = clinical_df.set_index(sample_id_column)

output_df = pd.concat([clinical_df, result], axis=1)
output_df = output_df.reset_index(drop=False)

output_df.to_csv(f"{GEO_ID}_preprocessed_ssgsea.csv", index=False)

# sanity check (there is 50 gene sets)
assert output_df.shape[1] == summary_data["data_summary"]["clinical_features"] + 50
